In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")
sns.set_palette(palette="muted")
plt.rcParams["figure.dpi"] = 500
plt.rcParams["savefig.dpi"] = 800
plt.rcParams["font.size"] = 12
plt.rcParams["lines.linewidth"] = 1.5

from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    RobustScaler,
    LabelEncoder,
    OneHotEncoder,
)
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, GridSearchCV
import catboost
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.impute import SimpleImputer
import scipy.stats as stats
import os

In [2]:
from pathlib import Path

BASE_DIR = Path.cwd().parent
DATA_PATH = BASE_DIR / "data" / "raw" / "train.csv"

data = pd.read_csv(DATA_PATH, sep=';')

In [3]:
def impute_column(df, target_col, feature_cols, model):
    """
    Заполняет только пропуски в target_col, используя model.
    Реальные значения не изменяются.
    """
    df = df.copy()

    missing_mask = df[target_col].isna()

    features_available_mask = df[feature_cols].notna().all(axis=1)

    rows_to_predict = missing_mask & features_available_mask

    if rows_to_predict.sum() > 0:
        predictions = model.predict(df.loc[rows_to_predict, feature_cols])
        df.loc[rows_to_predict, target_col] = predictions

    still_missing = (df[target_col].isna()).sum()
    print(
        f"{target_col}: заполнено {rows_to_predict.sum()}, "
        f"осталось NaN (нет донор-фич): {still_missing}"
    )

    return df

In [ ]:
import optuna
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error


def objective(trial, X, y, groups, feature_cols, cat_features=None):
    params = {
        "iterations": trial.suggest_int("iterations", 200, 1000),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 0.2, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 10.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "loss_function": "MAE",
        "random_seed": 42,
        "verbose": False,
        "allow_writing_files": False,
    }

    gkf = GroupKFold(n_splits=5)
    fold_scores = []

    for train_idx, valid_idx in gkf.split(X, y, groups):
        X_tr, X_val = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[valid_idx]

        model = CatBoostRegressor(**params)
        model.fit(
            X_tr,
            y_tr,
            cat_features=cat_features,
            eval_set=(X_val, y_val),
            early_stopping_rounds=50,
            verbose=False,
        )

        preds = model.predict(X_val)
        fold_scores.append(mean_absolute_error(y_val, preds))

    return np.mean(fold_scores)

Восстановим `DTC` по известным и наиболее полным данным: `GR`, `Z_LOC`

In [10]:
target_col = "DTC"
feature_cols = ["GR", "Z_LOC"]
cat_features = None

df_clean = data.dropna(subset=[target_col] + feature_cols + ["WELL"])
X = df_clean[feature_cols]
y = df_clean[target_col]
groups = df_clean["WELL"]

study = optuna.create_study(direction="minimize")
study.optimize(
    lambda trial: objective(trial, X, y, groups, feature_cols, cat_features),
    n_trials=5,
    show_progress_bar=True,
)

print("Лучшие параметры:", study.best_params)
print("Лучший MAE (CV):", study.best_value)

best_params = study.best_params
best_params.update({"loss_function": "MAE", "random_seed": 42, "verbose": False})

final_model_dtc = CatBoostRegressor(**best_params)
final_model_dtc.fit(X, y, cat_features=cat_features, verbose=False)

data_imputed = data.copy()

data_imputed = impute_column(
    data_imputed,
    target_col="DTC",
    feature_cols=["GR", "Z_LOC"],
    model=final_model_dtc,
)

final_model_dtc.save_model("catboost_final_model_dtc.cbm")

[I 2026-07-22 13:27:49,653] A new study created in memory with name: no-name-8b35174d-3623-453f-a86c-eeaf46dbfd70


  0%|          | 0/5 [00:00<?, ?it/s]

[I 2026-07-22 13:28:18,912] Trial 0 finished with value: 11.996764948532936 and parameters: {'iterations': 532, 'depth': 4, 'learning_rate': 0.19657828367328337, 'l2_leaf_reg': 5.48902767996224, 'subsample': 0.88369120028123, 'random_strength': 0.0034779637716235248}. Best is trial 0 with value: 11.996764948532936.
[I 2026-07-22 13:30:08,928] Trial 1 finished with value: 12.021385143838051 and parameters: {'iterations': 584, 'depth': 6, 'learning_rate': 0.009780542290983196, 'l2_leaf_reg': 0.022472770676135848, 'subsample': 0.6788879132561212, 'random_strength': 0.01567031444058853}. Best is trial 0 with value: 11.996764948532936.
[I 2026-07-22 13:30:39,328] Trial 2 finished with value: 12.006227585580515 and parameters: {'iterations': 993, 'depth': 7, 'learning_rate': 0.08458400500666648, 'l2_leaf_reg': 6.918414766816212, 'subsample': 0.6285768402514187, 'random_strength': 0.1334720452227917}. Best is trial 0 with value: 11.996764948532936.
[I 2026-07-22 13:32:33,709] Trial 3 finished

In [11]:
final_model_dtc = CatBoostRegressor()
final_model_dtc.load_model("catboost_final_model_dtc.cbm")

Восстановим `RHOB` по известным и наиболее полным данным: `GR`, `Z_LOC`, `DTC`

In [ ]:
target_col = "RHOB"
feature_cols = ["GR", "Z_LOC", 'DTC']
cat_features = None

df_clean = data.dropna(subset=[target_col] + feature_cols + ["WELL"])
X = df_clean[feature_cols]
y = df_clean[target_col]
groups = df_clean["WELL"]

study = optuna.create_study(direction="minimize")
study.optimize(
    lambda trial: objective(trial, X, y, groups, feature_cols, cat_features),
    n_trials=5,
    show_progress_bar=True,
)

print("Лучшие параметры:", study.best_params)
print("Лучший MAE (CV):", study.best_value)

best_params = study.best_params
best_params.update({"loss_function": "MAE", "random_seed": 42, "verbose": False})

final_model_rhob = CatBoostRegressor(**best_params)
final_model_rhob.fit(X, y, cat_features=cat_features, verbose=False)

data_imputed = impute_column(
    data_imputed,
    target_col="RHOB",
    feature_cols=["GR", "Z_LOC", "DTC"],
    model=final_model_rhob,
)

final_model_rhob.save_model("catboost_final_model_rhob.cbm")

final_model_rhob = CatBoostRegressor()
final_model_rhob.load_model("catboost_final_model_rhob.cbm")

Восстановим `NPHI` по известным и наиболее полным данным: `GR`, `Z_LOC`, `DTC`, `RHOB`

In [ ]:
target_col = "NPHI"
feature_cols = ["Z_LOC", "GR", 'DTC', 'RHOB']
cat_features = None

df_clean = data.dropna(subset=[target_col] + feature_cols + ["WELL"])
X = df_clean[feature_cols]
y = df_clean[target_col]
groups = df_clean["WELL"]

study = optuna.create_study(direction="minimize")
study.optimize(
    lambda trial: objective(trial, X, y, groups, feature_cols, cat_features),
    n_trials=5,
    show_progress_bar=True,
)

print("Лучшие параметры:", study.best_params)
print("Лучший MAE (CV):", study.best_value)

best_params = study.best_params
best_params.update({"loss_function": "MAE", "random_seed": 42, "verbose": False})

final_model_nphi = CatBoostRegressor(**best_params)
final_model_nphi.fit(X, y, cat_features=cat_features, verbose=False)

data_imputed = impute_column(
    data_imputed,
    target_col="NPHI",
    feature_cols=["GR", "Z_LOC", "DTC", "RHOB"],
    model=final_model_nphi,
)

final_model_nphi.save_model("catboost_final_model_nphi.cbm")
final_model_nphi = CatBoostRegressor()
final_model_nphi.load_model("catboost_final_model_nphi.cbm")

In [ ]:
target_col = "DTS"
feature_cols = ["Z_LOC", "GR", "DTC", "RHOB", "NPHI"]
cat_features = None

df_clean = data.dropna(subset=[target_col] + feature_cols + ["WELL"])
X = df_clean[feature_cols]
y = df_clean[target_col]
groups = df_clean["WELL"]

study = optuna.create_study(direction="minimize")
study.optimize(
    lambda trial: objective(trial, X, y, groups, feature_cols, cat_features),
    n_trials=5,
    show_progress_bar=True,
)

print("Лучшие параметры:", study.best_params)
print("Лучший MAE (CV):", study.best_value)

best_params = study.best_params
best_params.update({"loss_function": "MAE", "random_seed": 42, "verbose": False})

final_model_dts = CatBoostRegressor(**best_params)
final_model_dts.fit(X, y, cat_features=cat_features, verbose=False)

data_imputed = impute_column(
    data_imputed,
    target_col="DTS",
    feature_cols=["Z_LOC", "GR", "DTC", "RHOB", "NPHI"],
    model=final_model_dts,
)

final_model_dts.save_model("catboost_final_model_dts.cbm")
final_model_dts = CatBoostRegressor()
final_model_dts.load_model("catboost_final_model_dts.cbm")

In [ ]:
target_col = "BS"
feature_cols = ["GR", "Z_LOC", "CALI", 'DTC']
cat_features = None

df_clean = data.dropna(subset=[target_col] + feature_cols + ["WELL"])
X = df_clean[feature_cols]
y = df_clean[target_col]
groups = df_clean["WELL"]

study = optuna.create_study(direction="minimize")
study.optimize(
    lambda trial: objective(trial, X, y, groups, feature_cols, cat_features),
    n_trials=5,
    show_progress_bar=True,
)

print("Лучшие параметры:", study.best_params)
print("Лучший MAE (CV):", study.best_value)

best_params = study.best_params
best_params.update({"loss_function": "MAE", "random_seed": 42, "verbose": False})

final_model_bs = CatBoostRegressor(**best_params)
final_model_bs.fit(X, y, cat_features=cat_features, verbose=False)

data_imputed = impute_column(
    data_imputed,
    target_col="BS",
    feature_cols=["GR", "Z_LOC", "CALI", "DTC"],
    model=final_model_bs,
)

final_model_bs.save_model("catboost_final_model_bs.cbm")
final_model_bs = CatBoostRegressor()
final_model_bs.load_model("catboost_final_model_bs.cbm")

In [ ]:
target_col = "CALI"
feature_cols = ["GR", "Z_LOC", "BS", "DTC"]
cat_features = None

df_clean = data.dropna(subset=[target_col] + feature_cols + ["WELL"])
X = df_clean[feature_cols]
y = df_clean[target_col]
groups = df_clean["WELL"]

study = optuna.create_study(direction="minimize")
study.optimize(
    lambda trial: objective(trial, X, y, groups, feature_cols, cat_features),
    n_trials=5,
    show_progress_bar=True,
)

print("Лучшие параметры:", study.best_params)
print("Лучший MAE (CV):", study.best_value)

best_params = study.best_params
best_params.update({"loss_function": "MAE", "random_seed": 42, "verbose": False})

final_model_cali = CatBoostRegressor(**best_params)
final_model_cali.fit(X, y, cat_features=cat_features, verbose=False)

data_imputed = impute_column(
    data_imputed,
    target_col="CALI",
    feature_cols=["GR", "Z_LOC", "BS", "DTC"],
    model=final_model_cali,
)

final_model_cali.save_model("catboost_final_model_cali.cbm")
final_model_cali = CatBoostRegressor()
final_model_cali.load_model("catboost_final_model_cali.cbm")

In [ ]:
d = {
    "amount": data_imputed.isna().sum(),
    "percent": data_imputed.isna().sum() / len(data_imputed),
}
missing = pd.DataFrame(d).sort_values("percent")

missing

Используем каскад: восстановим оставшиеся пропуски с помощью наиболее заполненных признаков

In [ ]:
target_col = "RHOB"
feature_cols = ["GR", "DEPTH_MD", 'DTC']
cat_features = None

df_clean = data.dropna(subset=[target_col] + feature_cols + ["WELL"])
X = df_clean[feature_cols]
y = df_clean[target_col]
groups = df_clean["WELL"]

study = optuna.create_study(direction="minimize")
study.optimize(
    lambda trial: objective(trial, X, y, groups, feature_cols, cat_features),
    n_trials=2,
    show_progress_bar=True,
)

print("Лучшие параметры:", study.best_params)
print("Лучший MAE (CV):", study.best_value)

best_params = study.best_params
best_params.update({"loss_function": "MAE", "random_seed": 42, "verbose": False})

final_model_rhob_cascade = CatBoostRegressor(**best_params)
final_model_rhob_cascade.fit(X, y, cat_features=cat_features, verbose=False)

data_imputed = impute_column(
    data_imputed,
    target_col="RHOB",
    feature_cols=["GR", "DEPTH_MD", "DTC"],
    model=final_model_rhob_cascade,
)

final_model_rhob_cascade.save_model("catboost_final_model_rhob_cascade.cbm")
final_model_rhob_cascade = CatBoostRegressor()
final_model_rhob_cascade.load_model("catboost_final_model_rhob_cascade.cbm")

In [ ]:
target_col = "NPHI"
feature_cols = ["GR", "DEPTH_MD", "DTC"]
cat_features = None

df_clean = data.dropna(subset=[target_col] + feature_cols + ["WELL"])
X = df_clean[feature_cols]
y = df_clean[target_col]
groups = df_clean["WELL"]

study = optuna.create_study(direction="minimize")
study.optimize(
    lambda trial: objective(trial, X, y, groups, feature_cols, cat_features),
    n_trials=3,
    show_progress_bar=True,
)

print("Лучшие параметры:", study.best_params)
print("Лучший MAE (CV):", study.best_value)

best_params = study.best_params
best_params.update({"loss_function": "MAE", "random_seed": 42, "verbose": False})

final_model_nphi_cascade = CatBoostRegressor(**best_params)
final_model_nphi_cascade.fit(X, y, cat_features=cat_features, verbose=False)

data_imputed = impute_column(
    data_imputed,
    target_col="NPHI",
    feature_cols=["GR", "DEPTH_MD", "DTC"],
    model=final_model_nphi_cascade,
)

final_model_nphi_cascade.save_model("catboost_final_model_nphi_cascade.cbm")
final_model_nphi_cascade = CatBoostRegressor()
final_model_nphi_cascade.load_model("catboost_final_model_nphi_cascade.cbm")

In [ ]:
target_col = "DTS"
feature_cols = ["DEPTH_MD", 'NPHI', "DTC"]
cat_features = None

df_clean = data.dropna(subset=[target_col] + feature_cols + ["WELL"])
X = df_clean[feature_cols]
y = df_clean[target_col]
groups = df_clean["WELL"]

study = optuna.create_study(direction="minimize")
study.optimize(
    lambda trial: objective(trial, X, y, groups, feature_cols, cat_features),
    n_trials=5,
    show_progress_bar=True,
)

print("Лучшие параметры:", study.best_params)
print("Лучший MAE (CV):", study.best_value)

best_params = study.best_params
best_params.update({"loss_function": "MAE", "random_seed": 42, "verbose": False})

final_model_dts_cascade = CatBoostRegressor(**best_params)
final_model_dts_cascade.fit(X, y, cat_features=cat_features, verbose=False)

data_imputed = impute_column(
    data_imputed,
    target_col="DTS",
    feature_cols=["DEPTH_MD", "NPHI", "DTC"],
    model=final_model_dts_cascade,
)

final_model_dts_cascade.save_model("catboost_final_model_dts_cascade.cbm")
final_model_dts_cascade = CatBoostRegressor()
final_model_dts_cascade.load_model("catboost_final_model_dts_cascade.cbm")

In [ ]:
target_col = "BS"
feature_cols = ["DEPTH_MD", "DTC"]
cat_features = None

df_clean = data.dropna(subset=[target_col] + feature_cols + ["WELL"])
X = df_clean[feature_cols]
y = df_clean[target_col]
groups = df_clean["WELL"]

study = optuna.create_study(direction="minimize")
study.optimize(
    lambda trial: objective(trial, X, y, groups, feature_cols, cat_features),
    n_trials=3,
    show_progress_bar=True,
)

print("Лучшие параметры:", study.best_params)
print("Лучший MAE (CV):", study.best_value)

best_params = study.best_params
best_params.update({"loss_function": "MAE", "random_seed": 42, "verbose": False})

final_model_bs_cascade = CatBoostRegressor(**best_params)
final_model_bs_cascade.fit(X, y, cat_features=cat_features, verbose=False)

data_imputed = impute_column(
    data_imputed,
    target_col="BS",
    feature_cols=["DEPTH_MD", "DTC"],
    model=final_model_bs_cascade,
)

final_model_bs_cascade.save_model("catboost_final_model_bs_cascade.cbm")
final_model_bs_cascade = CatBoostRegressor()
final_model_bs_cascade.load_model("catboost_final_model_bs_cascade.cbm")

In [ ]:
target_col = "CALI"
feature_cols = ["DTS", "BS", "DTC"]
cat_features = None

df_clean = data.dropna(subset=[target_col] + feature_cols + ["WELL"])
X = df_clean[feature_cols]
y = df_clean[target_col]
groups = df_clean["WELL"]

study = optuna.create_study(direction="minimize")
study.optimize(
    lambda trial: objective(trial, X, y, groups, feature_cols, cat_features),
    n_trials=3,
    show_progress_bar=True,
)

print("Лучшие параметры:", study.best_params)
print("Лучший MAE (CV):", study.best_value)

best_params = study.best_params
best_params.update({"loss_function": "MAE", "random_seed": 42, "verbose": False})

final_model_cali_cascade = CatBoostRegressor(**best_params)
final_model_cali_cascade.fit(X, y, cat_features=cat_features, verbose=False)

data_imputed = impute_column(
    data_imputed,
    target_col="CALI",
    feature_cols=["DTS", "BS", "DTC"],
    model=final_model_cali_cascade,
)

final_model_cali_cascade.save_model("catboost_final_model_cali_cascade.cbm")
final_model_cali_cascade = CatBoostRegressor()
final_model_cali_cascade.load_model("catboost_final_model_cali_cascade.cbm")

In [ ]:
d = {
    "amount": data_imputed.isna().sum(),
    "percent": data_imputed.isna().sum() / len(data_imputed),
}
missing = pd.DataFrame(d).sort_values("percent")

missing

### Реализуем препроцессинг

**Метаданные**

| Столбец | Описание | Способ препроцессинга |
| :--- | :--- | :--- |
| `WELL` | Название скважины | drop, but the way to split the data
| `DEPTH_MD` | Измеренная глубина (Measured Depth) | StScaler
| `X_LOC` | Координата X в системе UTM | probably drop
| `Y_LOC` | Координата Y в системе UTM | probably drop
| `Z_LOC` | Абсолютная глубина (TVD) | StScaler
| `GROUP` | Литостратиграфическая группа по классификации NPD | enc
| `FORMATION` | Литостратиграфическая формация по классификации NPD | drop

**Каротажные кривые**

Данные содержат кривые ГИС, например:

| Кривая | Расшифровка | Способ препроцессинга |
| :--- | :--- | :--- |
| `BS` | Диаметр долота (Bit Size) | corr with cali
| `CALI` | Каверномер (Caliper) | StSc corr with bs
| `RDEP` | Глубокое удельное сопротивление (Deep Resistivity) | log or drop
| `RHOB` | Объёмная плотность (Bulk Density) | basis
| `GR` | Гамма-каротаж (сырые данные) | basis
| `SGR` | Спектральный гамма-каротаж (Spectral Gamma Ray) | probably drop
| `RMED` | Среднее удельное сопротивление (Medium Resistivity) | log or drop
| `ROP` | Скорость проходки (Rate of Penetration) | probably drop
| `NPHI` | Нейтронная пористость (Neutron Porosity) | StSc basis
| `PEF` | Фактор фотоэлектрического поглощения (Photoelectric Absorption Factor) | drop
| `RSHA` | Малоглубинное удельное сопротивление (Shallow Resistivity) | log or drop
| `DTS` | Акустический каротаж (поперечная волна, Shear Slowness) | imputataion target
| `DTC` | Акустический каротаж (продольная волна, Compressional Slowness) | StSc basis

**Данные интерпретации**

| Столбец | Описание | Способ препроцессинга |
| :--- | :--- | :--- |
| `FORCE_2020_LITHOFACIES_LITHOLOGY` | Метка класса литологии | target
| `FORCE_2020_LITHOFACIES_CONFIDENCE` | Степень достоверности интерпретации литологии (1 — высокая, 2 — средняя, 3 — низкая) | очистки обучающего набора или взвешивания потерь при обучении.



In [13]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

In [12]:
def new_features(df):
    df = df.copy()

    if "RHOB" in df.columns and "DTS" in df.columns:
        df["S_IMPEDANCE"] = 1e6 * df["RHOB"] / df["DTS"]

    if "RHOB" in df.columns and "DTC" in df.columns:
        df["P_IMPEDANCE"] = 1e6 * df["RHOB"] / df["DTC"]

    if "RHOB" in df.columns and "DTS" in df.columns:
        df["G_SHEAR_MODULUS"] = 1e6 * df["RHOB"] / df["DTS"] ** 2

    if "RHOB" in df.columns and "DTC" in df.columns and "DTS" in df.columns:
        df["BULK_MODULUS"] = (
            1e6 * df["RHOB"] * (1 / (df["DTC"] ** 2) - (4 / 3) / (df["DTS"] ** 2))
        )

    if "RHOB" in df.columns:
        df["TOC"] = 154.497 / df["RHOB"] - 57.261

    return df

In [ ]:
class FeatureEngineering(BaseEstimator, TransformerMixin):
    def __init__(self, cols_to_drop=None, cols_to_log=None):
        self.cols_to_drop = cols_to_drop
        self.cols_to_log = cols_to_log

    def fit(self, df, y=None):
        return self

    def transform(self, df):
        df_prep = df.copy()

        if self.cols_to_drop:
            df_prep = df_prep.drop(columns=self.cols_to_drop, errors='ignore')

        df_prep = new_features(df_prep)

        if self.cols_to_log:
            for col in self.cols_to_log:
                if col in df_prep.columns:
                    df_prep[col] = np.log10(df_prep[col] * 10 + 1)
                    
        return df_prep

In [ ]:
cols_to_drop = [
    "X_LOC",
    "Y_LOC",
    "FORMATION",
    "RDEP",
    "SGR",
    "RMED",
    "ROP",
    "PEF",
    "RSHA",
    "RXO",
    "MUDWEIGHT",
    "DCAL",
    "RMIC",
    "ROPA",
    "WELL",
    "FORCE_2020_LITHOFACIES_CONFIDENCE",
]
cols_to_log = ['RDEP', 'RMED', 'RSHA']

Реализация с помощью RF

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier

numeric_transformer = Pipeline(
    [
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        ("scaler", StandardScaler()),
    ]
)

pipeline_rf = Pipeline(
    [
        ("feature", FeatureEngineering(cols_to_drop, cols_to_log)),
        (
            "preprocessor",
            ColumnTransformer(
                [
                    (
                        "num",
                        numeric_transformer,
                        make_column_selector(dtype_include="number"),
                    ),
                    (
                        "cat",
                        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        make_column_selector(dtype_include=object),
                    ),
                ],
                remainder="passthrough",
                verbose_feature_names_out=False,
            ),
        ),
        (
            "model",
            RandomForestClassifier(random_state=42, n_jobs=-1)
        ),
    ]
)

pipeline_rf.named_steps["preprocessor"].set_output(transform="pandas")